# Day 041 — Exercise 3: extract_code

**What you'll build:** `extract_code(response) -> str` — parse a raw LLM response and return only the Python code, stripping markdown fences.

**Why it matters:** LLMs wrap code in markdown fences like `\`\`\`python ... \`\`\``. You cannot exec that directly — you need the code inside. `extract_code` uses `re.search` with `re.DOTALL` to match the content across newlines. It tries three patterns in order: `\`\`\`python`, plain `\`\`\``, then falls back to the raw response.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import re
import ollama
import pandas as pd
import io


RETAIL_CSV = (
    'order_id,product,category,region,price,quantity\n'
    '1,Widget,Electronics,North,25.0,10\n'
    '2,Gadget,Electronics,South,150.0,3\n'
    '3,Widget,Electronics,South,25.0,5\n'
    '4,Doohickey,Accessories,East,8.0,50\n'
    '5,Gadget,Electronics,East,150.0,7\n'
    '6,Widget,Electronics,East,25.0,4\n'
    '7,Doohickey,Accessories,North,8.0,20\n'
    '8,Gadget,Electronics,North,150.0,2\n'
    '9,Widget,Electronics,West,25.0,6\n'
    '10,Doohickey,Accessories,South,8.0,15\n'
    '11,Thingamajig,Accessories,North,200.0,1\n'
    '12,Thingamajig,Accessories,East,200.0,4'
)
SALES_DF = pd.read_csv(io.StringIO(RETAIL_CSV))
SALES_DF['revenue'] = SALES_DF['price'] * SALES_DF['quantity']

## Your Implementation

In [ ]:
import re

def extract_code(response: str) -> str:
    """
    Extract Python code from an LLM response.

    Try in order:
    1. re.search(r'```python\\s*(.*?)```', response, re.DOTALL)
    2. re.search(r'```\\s*(.*?)```', response, re.DOTALL)
    3. Fall back: return response.strip()

    In all cases, return match.group(1).strip() or response.strip().

    Returns:
        str — the extracted (and stripped) Python code
    """
    # TODO: try pattern 1: r'```python\s*(.*?)```' with re.DOTALL
    # TODO: if matched, return match.group(1).strip()
    # TODO: try pattern 2: r'```\s*(.*?)```' with re.DOTALL
    # TODO: if matched, return match.group(1).strip()
    # TODO: fall back: return response.strip()
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: function defined
    try:
        assert 'extract_code' in globals()
        passed += 1; print('\u2705 Check 1: extract_code is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: extracts from ```python...``` block
    try:
        _resp = '```python\nresult = df.shape[0]\n```'
        _code = extract_code(_resp)
        assert _code == 'result = df.shape[0]', \
            f'expected "result = df.shape[0]", got {repr(_code)}'
        passed += 1; print('\u2705 Check 2: extracts from ```python block')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: extracts from plain ``` block
    try:
        _resp2 = '```\nresult = 42\n```'
        _code2 = extract_code(_resp2)
        assert _code2 == 'result = 42', \
            f'expected "result = 42", got {repr(_code2)}'
        passed += 1; print('\u2705 Check 3: extracts from plain ``` block')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: handles multi-line code blocks correctly
    try:
        _multi = '```python\ntotals = df.groupby("product")["revenue"].sum()\nresult = totals.idxmax()\n```'
        _extracted = extract_code(_multi)
        assert 'totals' in _extracted and 'result' in _extracted, \
            'multi-line block not extracted correctly'
        passed += 1; print('\u2705 Check 4: handles multi-line code block')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: falls back to stripped raw text when no fence found
    try:
        _raw = '  result = df.shape[0]  '
        _fallback = extract_code(_raw)
        assert _fallback == 'result = df.shape[0]', \
            f'fallback should strip whitespace, got {repr(_fallback)}'
        passed += 1; print('\u2705 Check 5: falls back to stripped raw text')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import re

def extract_code(response: str) -> str:
    fence = '`' * 3
    match = re.search(fence + r'python\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(fence + r'\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()
```

</details>